In [ ]:
from pathlib import Path

from sc4_reading_func import binary_to_clean_txt
from sc4_s02_txtF2parquet import reading__measurements_file
from parquet_reshaping_lvl0 import pq_reshaping_lvl0


def run_pipeline(
    binary_dir,
    txt_dir,
    mearem_dir,
    lvl0_dir):
    """
    Complete SC4 processing pipeline.

    binary_dir
        Folder containing binary files.

    txt_dir
        Folder where txt files will be written.

    mearem_dir
        Folder where mearem parquet files will be written.

    lvl0_dir
        Folder where lvl0 parquet files will be written.
    """

    binary_dir = Path(binary_dir)
    txt_dir = Path(txt_dir)
    mearem_dir = Path(mearem_dir)
    lvl0_dir = Path(lvl0_dir)

    txt_dir.mkdir(exist_ok=True, parents=True)
    mearem_dir.mkdir(exist_ok=True, parents=True)
    lvl0_dir.mkdir(exist_ok=True, parents=True)

    binary_files = sorted(binary_dir.glob("*.??_"))

    print(f"Found {len(binary_files)} binary files\n")

    for i, binary_file in enumerate(binary_files, start=1):

        print("="*70)
        print(f"[{i}/{len(binary_files)}] {binary_file.name}")

        try:


            txt_file = binary_to_clean_txt(
                binary_file,
                output_dir=txt_dir
            )


            mearem_file = reading__measurements_file(
                txt_file)



            lvl0_df, lvl0_file = pq_reshaping_lvl0(
                mearem_file,
                output_dir=lvl0_dir
            )

            print("✓ Finished")

        except Exception as e:

            print("✗ Failed")
            print(e)

    print("\nPipeline completed.")

In [15]:
run_pipeline(
    binary_dir="/Users/dal674840/Downloads/binary_files",
    txt_dir="/Users/dal674840/Downloads/txt_files",
    mearem_dir="/Users/dal674840/Downloads/mearem_parquet",
    lvl0_dir="/Users/dal674840/Downloads/lvl0_parquet"
)

Found 1 binary files

[1/1] cg01001a00.26_
✗ Failed
[Errno 2] No such file or directory: 'rxtools_exec.sh'

Pipeline completed.


In [18]:
from pathlib import Path
import shutil

from sc4_reading_func import binary_to_clean_txt
from sc4_s02_txtF2parquet import reading__measurements_file
from parquet_reshaping_lvl0 import pq_reshaping_lvl0


def run_pipeline(
    binary_dir,
    txt_dir,
    mearem_dir,
    lvl0_dir,
):
    """
    Complete SC4 processing pipeline

    Binary
        ↓
    TXT
        ↓
    Measurement Parquet
        ↓
    Level-0 Parquet

    Parameters
    ----------
    binary_dir : str or Path
        Directory containing binary files.

    txt_dir : str or Path
        Directory for cleaned txt files.

    mearem_dir : str or Path
        Directory for measurement parquet files.

    lvl0_dir : str or Path
        Directory for level0 parquet files.
    """

    binary_dir = Path(binary_dir)
    txt_dir = Path(txt_dir)
    mearem_dir = Path(mearem_dir)
    lvl0_dir = Path(lvl0_dir)

    txt_dir.mkdir(parents=True, exist_ok=True)
    mearem_dir.mkdir(parents=True, exist_ok=True)
    lvl0_dir.mkdir(parents=True, exist_ok=True)

    binary_files = sorted(binary_dir.glob("*.26_"))

    print(f"\nFound {len(binary_files)} binary files.\n")

    successful = 0
    failed = []

    for i, binary_file in enumerate(binary_files, start=1):

        print("=" * 80)
        print(f"[{i}/{len(binary_files)}] {binary_file.name}")

        try:

            ############################################################
            # STEP 1
            # Binary -> TXT
            ############################################################

            txt_file = binary_to_clean_txt(
                binary_file,
                output_dir=txt_dir
            )

            txt_file = Path(txt_file)

            print(f"TXT created:")
            print(txt_file)

            ############################################################
            # STEP 2
            # TXT -> Measurement parquet
            ############################################################

            mearem_df, mearem_file = reading__measurements_file(
                str(txt_file)
            )

            mearem_file = Path(mearem_file)

            ############################################################
            # Move parquet to mearem directory
            ############################################################

            destination = mearem_dir / mearem_file.name

            if mearem_file != destination:

                shutil.move(
                    str(mearem_file),
                    str(destination)
                )

                mearem_file = destination

            print(f"Measurement parquet:")
            print(mearem_file)

            ############################################################
            # STEP 3
            # Measurement parquet -> Level0 parquet
            ############################################################

            lvl0_df, lvl0_file = pq_reshaping_lvl0(
                mearem_file,
                output_dir=lvl0_dir
            )

            print(f"Level0 parquet:")
            print(lvl0_file)

            successful += 1

            print("✓ Success")

        except Exception as e:

            failed.append(binary_file.name)

            print("✗ Failed")
            print(e)

    print("\n" + "=" * 80)
    print("Pipeline completed")
    print("=" * 80)

    print(f"Successful : {successful}")
    print(f"Failed     : {len(failed)}")

    if failed:

        print("\nFailed files:")

        for file in failed:

            print(file)

          